In [191]:
import pandas as pd
import numpy as np
#imports

In [192]:
df_order_items=pd.read_csv("https://raw.githubusercontent.com/alvarolucio2007/Atividade-Final-Ciencia-De-Dados/refs/heads/main/dados/olist_order_items_dataset.csv")
df_orders=pd.read_csv("https://raw.githubusercontent.com/alvarolucio2007/Atividade-Final-Ciencia-De-Dados/refs/heads/main/dados/olist_orders_dataset.csv")
df_products=pd.read_csv("https://raw.githubusercontent.com/alvarolucio2007/Atividade-Final-Ciencia-De-Dados/refs/heads/main/dados/olist_order_items_dataset.csv")
df_order_reviews=pd.read_csv("https://raw.githubusercontent.com/alvarolucio2007/Atividade-Final-Ciencia-De-Dados/refs/heads/main/dados/olist_order_reviews_dataset.csv")

In [193]:
df_order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [194]:
#[IA] Criacao de Pipeline de verificação para df_order_items
def auditar_order_items(df: pd.DataFrame) -> pd.DataFrame:
    """
    Roda checagens de qualidade e devolve um DataFrame de relatório.
    NÃO altera o df original.
    """
    checks = []
    n = len(df)

    def add(nome, qtd, total=n, detalhe=""):
        checks.append({
            "check": nome,
            "problemas": int(qtd),
            "pct (%)": round(100 * qtd / total, 4) if total else 0.0,
            "status": "OK" if qtd == 0 else "FALHA",
            "obs": detalhe,
        })

    # ---------- 1. Nulos ----------
    for col in df.columns:
        add(f"nulos em {col}", df[col].isna().sum())

    # ---------- 2. IDs (object) ----------
    for col in ["order_id", "product_id", "seller_id"]:
        s = df[col].astype(str)
        add(f"{col} vazio/whitespace", s.str.strip().eq("").sum())
        add(f"{col} com espaço nas bordas", (s != s.str.strip()).sum())
        # Olist usa hash 32-hex; se não for Olist, remova esta linha
        add(f"{col} fora do padrão 32-hex",
            (~s.str.match(r"^[0-9a-f]{32}$")).sum())

    # ---------- 3. order_item_id ----------
    add("order_item_id <= 0", (df["order_item_id"] <= 0).sum())

    # ---------- 4. Chave composta ----------
    add("chave (order_id, order_item_id) duplicada",
        df.duplicated(subset=["order_id", "order_item_id"]).sum())

    # ---------- 5. shipping_limit_date ----------
    dt = pd.to_datetime(df["shipping_limit_date"], errors="coerce",
                        format="%Y-%m-%d %H:%M:%S")
    add("shipping_limit_date inválida (não parseia)", dt.isna().sum())
    add("shipping_limit_date no futuro", (dt > pd.Timestamp.today()).sum())
    add("shipping_limit_date < 2000", (dt < pd.Timestamp("2000-01-01")).sum())
    # formato estrito (dd/mm/aaaa? sem hora? etc.)
    add("shipping_limit_date fora do padrão YYYY-MM-DD HH:MM:SS",
        (dt.notna() & (df["shipping_limit_date"]
                       .astype(str)
                       .str.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")
                       .fillna(False))).eq(False).sum() - dt.isna().sum())

    # ---------- 6. price ----------
    add("price < 0", (df["price"] < 0).sum())
    add("price == 0", (df["price"] == 0).sum(), detalhe="informativo (promo)")
    add("price com > 2 decimais",
        (~np.isclose(df["price"], df["price"].round(2), atol=1e-9)).sum())
    q1, q3 = df["price"].quantile([.25, .75])
    lim = q3 + 3 * (q3 - q1)
    add(f"price outlier > {lim:.2f} (Q3+3IQR)",
      (df["price"] > lim).sum(), detalhe="informativo")

    # ---------- 7. freight_value ----------
    add("freight_value < 0", (df["freight_value"] < 0).sum())
    add("freight_value == 0", (df["freight_value"] == 0).sum(),
        detalhe="informativo (frete grátis)")
    add("freight_value com > 2 decimais",
        (~np.isclose(df["freight_value"], df["freight_value"].round(2), atol=1e-9)).sum())
    q1, q3 = df["freight_value"].quantile([.25, .75])
    lim = q3 + 3 * (q3 - q1)
    add(f"freight_value outlier > {lim:.2f} (Q3+3IQR)",
        (df["freight_value"] > lim).sum(), detalhe="informativo")

    # ---------- 8. Consistência de agrupamento ----------
    cont = df.groupby("order_id")["order_item_id"].agg(["min", "max", "count"])
    add("order_id cujo order_item_id não começa em 1",
        (cont["min"] != 1).sum(), total=len(cont), detalhe="informativo")
    add("order_id com itens não sequenciais (buracos)",
        ((cont["max"] - cont["min"] + 1) != cont["count"]).sum(),
        total=len(cont), detalhe="informativo")

    return pd.DataFrame(checks)



In [195]:
#[IA] Criação de Pipeline de limpeza para df_order_items
def limpar_order_items(df: pd.DataFrame, manter_outliers=True) -> pd.DataFrame:
    n0 = len(df)
    log = []
    df = df.copy()

    # --- 1. IDs: strip ---
    for col in ["order_id", "product_id", "seller_id"]:
        antes = df[col].astype(str)
        df[col] = antes.str.strip()
        n = (antes != df[col]).sum()
        if n: log.append(f"{col}: {n} linhas com espaço removido")

    # --- 2. order_item_id: int ---
    df["order_item_id"] = pd.to_numeric(df["order_item_id"], errors="coerce")

    # --- 3. shipping_limit_date: datetime ---
    df["shipping_limit_date"] = pd.to_datetime(
        df["shipping_limit_date"], errors="coerce",
        format="%Y-%m-%d %H:%M:%S"
    )
    n = df["shipping_limit_date"].isna().sum()
    if n: log.append(f"shipping_limit_date inválida: {n}")

    # --- 4. Numéricos: round 2 ---
    for col in ["price", "freight_value"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").round(2)

    # --- 5. Máscara de remoção (regras de negócio) ---
    mask_ruim = (
        df["order_id"].eq("") | df["product_id"].eq("") | df["seller_id"].eq("")
        | df["order_item_id"].isna() | (df["order_item_id"] <= 0)
        | df["shipping_limit_date"].isna()
        | df["shipping_limit_date"].gt(pd.Timestamp.today())
        | df["shipping_limit_date"].lt(pd.Timestamp("2010-01-01"))
        | df["price"].isna() | (df["price"] < 0)
        | df["freight_value"].isna() | (df["freight_value"] < 0)
    )
    if not manter_outliers:
        for col in ["price", "freight_value"]:
            q1, q3 = df[col].quantile([.25, .75])
            lim = q3 + 3 * (q3 - q1)
            mask_ruim |= df[col] > lim

    n = mask_ruim.sum()
    if n: log.append(f"linhas removidas por regra de negócio: {n}")
    df = df[~mask_ruim]

    # --- 6. Dedup pela chave de negócio ---
    antes = len(df)
    df = df.drop_duplicates(subset=["order_id", "order_item_id"], keep="last")
    if antes - len(df): log.append(f"duplicatas (order_id, order_item_id): {antes-len(df)}")

    # --- 7. Tipos finais ---
    df["order_item_id"] = df["order_item_id"].astype("int32")
    df["price"] = df["price"].astype("float32")
    df["freight_value"] = df["freight_value"].astype("float32")

    df = df.reset_index(drop=True)

    print(f"Linhas: {n0} -> {len(df)} (removidas {n0-len(df)}, "
          f"{100*(n0-len(df))/n0:.2f}%)")
    for l in log: print("  •", l)
    return df


In [196]:
rel=auditar_order_items(df_order_items)
print(rel.to_string(index=False))
print(f"\nFalhas: {(rel['status']=='FALHA').sum()} | "
      f"OK: {(rel['status']=='OK').sum()}")
#Dá pra ver que a maioria dos produtos não possui problemas,
#apenas possuindo outliers e afins, o que não necessariamente é ruim, podendo ser
#promoções, clientes ricos, etc.

                                                 check  problemas  pct (%) status                        obs
                                     nulos em order_id          0   0.0000     OK                           
                                nulos em order_item_id          0   0.0000     OK                           
                                   nulos em product_id          0   0.0000     OK                           
                                    nulos em seller_id          0   0.0000     OK                           
                          nulos em shipping_limit_date          0   0.0000     OK                           
                                        nulos em price          0   0.0000     OK                           
                                nulos em freight_value          0   0.0000     OK                           
                             order_id vazio/whitespace          0   0.0000     OK                           
                   

In [197]:
order_items_limpo=limpar_order_items(df_order_items)
# Nenhuma linha removida: o dataset já estava limpo. Os outliers presentes,
# como mencionado antes, não representam um problema a ser corrigido.

Linhas: 112650 -> 112650 (removidas 0, 0.00%)


In [198]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [199]:
df_orders['order_status'].unique()

array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'], dtype=object)

In [200]:
#[IA] Criação de Pipeline de verificação para df_orders
import pandas as pd
import numpy as np

# Status válidos MANUALMENTE EDITADOS POIS ESTAVAM ERRADOS, VISTOS NO UNIQUE ACIMA.
STATUS_VALIDOS = {
    'delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'
}

# piso de sanidade temporal (você escolheu 2000 — ok, registra no diário)
DATA_MIN = pd.Timestamp("2000-01-01")

COLS_DATA = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]


def auditar_orders(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add(nome, qtd, total=None, obs=""):
        total = len(df) if total is None else total
        checks.append({
            "check": nome,
            "problemas": int(qtd),
            "pct (%)": round(100 * qtd / total, 4) if total else 0.0,
            "status": "OK" if qtd == 0 else "FALHA",
            "obs": obs,
        })

    # ---------- 1. Nulos brutos (só contagem, nulo aqui não é erro por si) ----------
    for col in df.columns:
        n = df[col].isna().sum()
        add(f"nulos em {col}", n, obs="informativo" if n else "")

    # ---------- 2. Domínio de status ----------
    add("order_status fora do domínio",
        (~df["order_status"].isin(STATUS_VALIDOS)).sum())

    # ---------- 3. Parse + range mínimo de cada data ----------
    dt = {}
    for col in COLS_DATA:
        dt[col] = pd.to_datetime(df[col], errors="coerce")
        invalido = df[col].notna() & dt[col].isna()
        add(f"{col} inválida (não parseia)", invalido.sum())
        add(f"{col} < 2000", (dt[col] < DATA_MIN).sum())

    p = dt["order_purchase_timestamp"]
    a = dt["order_approved_at"]
    c = dt["order_delivered_carrier_date"]
    d = dt["order_delivered_customer_date"]
    e = dt["order_estimated_delivery_date"]

    # ---------- 4. Ordem lógica entre os timestamps ----------
    add("approved < purchase",          (a < p).sum())
    add("carrier   < purchase",         (c < p).sum())
    add("carrier   < approved (quando approved presente)", ((a.notna()) & (c < a)).sum())
    add("customer  < carrier",          (d < c).sum())
    add("customer  < purchase",         (d < p).sum())
    add("customer  < approved (quando approved presente)", ((a.notna()) & (d < a)).sum())

    # ---------- 5. Nulos CONDICIONAIS ao status ----------
    precisa_approved = df["order_status"].isin(
        {"approved", "invoiced", "processing", "shipped", "delivered"}
    )
    add("approved nulo mas status exige", (precisa_approved & a.isna()).sum())

    entregue = df["order_status"].eq("delivered")
    add("carrier nulo mas status=delivered",  (entregue & c.isna()).sum())
    add("customer nulo mas status=delivered", (entregue & d.isna()).sum())

    # ---------- 6. Estimated x customer (FLAG, não erro) ----------
    valid = d.notna() & e.notna()
    atrasado = valid & (d > e)
    adiantado = valid & (d < e)
    add("entrega após estimativa (atraso)", atrasado.sum(), obs="informativo (negócio)")
    add("entrega antes da estimativa",       adiantado.sum(), obs="informativo (negócio)")

    # margem estatística do atraso — só pra enxergar cauda
    diff = (d - e).dt.days
    if diff.notna().any():
        q1, q3 = diff.quantile([.25, .75])
        lim = q3 + 3 * (q3 - q1)
        add(f"atraso > {lim:.0f} dias (Q3+3IQR)",
            (diff > lim).sum(), obs="informativo (outlier, NÃO remover)")

    return pd.DataFrame(checks)


In [201]:
def limpar_orders(df: pd.DataFrame) -> pd.DataFrame:
    n0 = len(df)
    log = []
    df = df.copy().reset_index(drop=True)   # normaliza índice logo de cara

    # --- Cast de datas ---
    for col in COLS_DATA:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    p = df["order_purchase_timestamp"]
    a = df["order_approved_at"]
    c = df["order_delivered_carrier_date"]
    d = df["order_delivered_customer_date"]
    e = df["order_estimated_delivery_date"]

    # --- Flags ANTES do filtro (índices ainda alinhados) ---
    df["flag_atraso"]        = d.notna() & e.notna() & (d > e)
    df["flag_entrega_antes"] = d.notna() & e.notna() & (d < e)
    df["flag_sem_aprovacao"] = a.isna()
    df["flag_sem_entrega"]   = d.isna()

    # --- Máscara de invalidez dura ---
    mask_ruim = (
        df["order_id"].isna() | df["order_id"].eq("")
        | df["customer_id"].isna() | df["customer_id"].eq("")
        | ~df["order_status"].isin(STATUS_VALIDOS)
        | p.isna() | (p < DATA_MIN)
        | (a < p)
        | (c < p)
        | (a.notna() & (c < a))
        | (d < c)
        | (d < p)
        | (df["order_status"].eq("delivered") & (c.isna() | d.isna()))
        | df["order_status"].isin({"approved", "invoiced", "processing", "shipped", "delivered"}) & a.isna()
    )

    n = int(mask_ruim.sum())
    if n:
        log.append(f"linhas removidas por invalidez dura: {n}")
    df = df[~mask_ruim].reset_index(drop=True)

    # --- Dedup pela PK ---
    antes = len(df)
    df = df.drop_duplicates(subset=["order_id"], keep="last").reset_index(drop=True)
    if antes - len(df):
        log.append(f"order_id duplicado: {antes - len(df)}")

    print(f"Linhas: {n0} -> {len(df)} (removidas {n0-len(df)}, "
          f"{100*(n0-len(df))/n0:.3f}%)")
    for l in log:
        print("  •", l)
    return df

In [202]:

rel = auditar_orders(df_orders)
print(rel.to_string(index=False))
print(f"\nFalhas: {(rel['status']=='FALHA').sum()} | OK: {(rel['status']=='OK').sum()}")
#Falhas verdadeiras (não informativas) em comparação de carrier,purchase e approved, e também carrier/costumer nulo com status=delivered

                                               check  problemas  pct (%) status                                obs
                                   nulos em order_id          0   0.0000     OK                                   
                                nulos em customer_id          0   0.0000     OK                                   
                               nulos em order_status          0   0.0000     OK                                   
                   nulos em order_purchase_timestamp          0   0.0000     OK                                   
                          nulos em order_approved_at        160   0.1609  FALHA                        informativo
               nulos em order_delivered_carrier_date       1783   1.7930  FALHA                        informativo
              nulos em order_delivered_customer_date       2965   2.9817  FALHA                        informativo
              nulos em order_estimated_delivery_date          0   0.0000     OK 

In [203]:
df_ok = limpar_orders(df_orders)
print(df_ok.dtypes)
#Limpeza de orders.

rel = auditar_orders(df_ok)
print(rel.to_string(index=False))
print(f"\nFalhas: {(rel['status']=='FALHA').sum()} | OK: {(rel['status']=='OK').sum()}")
#Verificação com o pipeline de auditação, apenas falhou em informativos, aonde a falha não necessariamente é ruim.

Linhas: 99441 -> 98036 (removidas 1405, 1.413%)
  • linhas removidas por invalidez dura: 1405
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
flag_atraso                                bool
flag_entrega_antes                         bool
flag_sem_aprovacao                         bool
flag_sem_entrega                           bool
dtype: object
                                               check  problemas  pct (%) status                                obs
                                   nulos em order_id          0   0.0000     OK                                   
                                nulos em customer_id          0   0.0000     OK       

In [204]:
df_order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [205]:
#[IA] Criação de pipeline de order_reviews.

DATA_MIN = pd.Timestamp("2000-01-01")   # mesmo piso que você já decidiu
SCORES_VALIDOS = {1, 2, 3, 4, 5}        # Olist usa 1-5; confirme com value_counts


def auditar_reviews(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add(nome, qtd, total=None, obs=""):
        total = len(df) if total is None else total
        checks.append({
            "check": nome,
            "problemas": int(qtd),
            "pct (%)": round(100 * qtd / total, 4) if total else 0.0,
            "status": "OK" if qtd == 0 else "FALHA",
            "obs": obs,
        })

    # ---------- 1. Nulos brutos (só contagem) ----------
    for col in df.columns:
        n = df[col].isna().sum()
        # title/message nulo é ESPERADO — marca informativo
        obs = "informativo (esperado)" if col in (
            "review_comment_title", "review_comment_message"
        ) and n else ""
        add(f"nulos em {col}", n, obs=obs)

    # ---------- 2. String vazia/whitespace disfarçada de dado ----------
    for col in ["review_comment_title", "review_comment_message"]:
        s = df[col].astype("string")            # preserva NaN
        vazio = s.notna() & s.str.strip().eq("")
        add(f"{col} vazio/whitespace (deveria ser nulo)", vazio.sum())

    # ---------- 3. IDs ----------
    for col in ["review_id", "order_id"]:
        s = df[col].astype(str)
        add(f"{col} vazio/whitespace", s.str.strip().eq("").sum())
        add(f"{col} com espaço nas bordas", (s != s.str.strip()).sum())

    # ---------- 4. review_score: domínio fechado ----------
    add("review_score <= 0",        (df["review_score"] <= 0).sum())
    add("review_score > 5",         (df["review_score"] > 5).sum())
    add("review_score fora de {1..5}",
        (~df["review_score"].isin(SCORES_VALIDOS)).sum())

    # ---------- 5. Datas ----------
    dt_c = pd.to_datetime(df["review_creation_date"], errors="coerce")
    dt_a = pd.to_datetime(df["review_answer_timestamp"], errors="coerce")

    add("review_creation_date inválida", df["review_creation_date"].notna().mul(dt_c.isna()).sum())
    add("review_answer_timestamp inválida", df["review_answer_timestamp"].notna().mul(dt_a.isna()).sum())
    add("review_creation_date < 2000", (dt_c < DATA_MIN).sum())
    add("review_answer_timestamp < 2000", (dt_a < DATA_MIN).sum())

    # ---------- 6. Ordem lógica: resposta >= criação ----------
    add("answer < creation", (dt_a < dt_c).sum())

    # ---------- 7. Duplicatas ----------
    add("review_id duplicado", df["review_id"].duplicated().sum())

    # Olist: teoricamente 1 review por pedido. Se aparecer mais, é flag
    # (o dataset original tem alguns casos, não é necessariamente erro).
    n_order_dup = df["order_id"].duplicated(keep=False).sum()
    add("order_id com mais de 1 review", n_order_dup,
        obs="informativo (Olist permite, checar)")

    return pd.DataFrame(checks)


In [206]:
# [IA] Pipeline de limpeza de df_order_reviews
def limpar_reviews(df: pd.DataFrame) -> pd.DataFrame:
    n0 = len(df)
    log = []
    df = df.copy().reset_index(drop=True)

    # --- Cast de datas ---
    df["review_creation_date"]    = pd.to_datetime(df["review_creation_date"], errors="coerce")
    df["review_answer_timestamp"] = pd.to_datetime(df["review_answer_timestamp"], errors="coerce")

    # --- Strip de IDs ---
    for col in ["review_id", "order_id"]:
        df[col] = df[col].astype("string").str.strip()

    # --- String vazia -> NaN em title/message (não remove, só normaliza) ---
    for col in ["review_comment_title", "review_comment_message"]:
        df[col] = df[col].astype("string")
        df[col] = df[col].mask(df[col].str.strip().eq(""))   # "" -> NA
        # se quiser preservar como string vazia em vez de NaN, comente a linha acima

    # --- score: coerce e cast ---
    df["review_score"] = pd.to_numeric(df["review_score"], errors="coerce").astype("Int8")

    # --- Máscara de invalidez dura (só o que é ERRO, não o que é nulo esperado) ---
    mask_ruim = (
        df["review_id"].isna() | df["review_id"].eq("")
        | df["order_id"].isna() | df["order_id"].eq("")
        | df["review_score"].isna()
        | ~df["review_score"].isin(SCORES_VALIDOS)
        | df["review_creation_date"].isna()
        | (df["review_creation_date"] < DATA_MIN)
        | df["review_answer_timestamp"].isna()
        | (df["review_answer_timestamp"] < DATA_MIN)
        | (df["review_answer_timestamp"] < df["review_creation_date"])  # resposta antes da criação
    )

    n = int(mask_ruim.sum())
    if n:
        log.append(f"linhas removidas por invalidez dura: {n}")
    df = df[~mask_ruim].reset_index(drop=True)
    df["flag_order_multi_review"] = df["order_id"].duplicated(keep=False) #flag do para >1 review por ordem.

    # --- Dedup pela PK ---
    antes = len(df)
    df = df.drop_duplicates(subset=["review_id"], keep="last").reset_index(drop=True)
    if antes - len(df):
        log.append(f"review_id duplicado: {antes - len(df)}")

    print(f"Linhas: {n0} -> {len(df)} (removidas {n0-len(df)}, "
          f"{100*(n0-len(df))/n0:.3f}%)")
    for l in log:
        print("  •", l)
    return df


In [207]:
#Auditoria de df_order_reviews
rel = auditar_reviews(df_order_reviews)
print(rel.to_string(index=False))
print(f"\nFalhas: {(rel['status']=='FALHA').sum()} | OK: {(rel['status']=='OK').sum()}")

                                                     check  problemas  pct (%) status                                 obs
                                        nulos em review_id          0   0.0000     OK                                    
                                         nulos em order_id          0   0.0000     OK                                    
                                     nulos em review_score          0   0.0000     OK                                    
                             nulos em review_comment_title      87656  88.3415  FALHA              informativo (esperado)
                           nulos em review_comment_message      58247  58.7025  FALHA              informativo (esperado)
                             nulos em review_creation_date          0   0.0000     OK                                    
                          nulos em review_answer_timestamp          0   0.0000     OK                                    
  review_comment_title v

In [208]:
#Limpeza de df_order_reviews
df_ok_reviews = limpar_reviews(df_order_reviews)
print(df_ok_reviews.dtypes)
#Como visto acima em review_id, apenas review_ids duplicados foram removidos.
#

Linhas: 99224 -> 98410 (removidas 814, 0.820%)
  • review_id duplicado: 814
review_id                  string[python]
order_id                   string[python]
review_score                         Int8
review_comment_title       string[python]
review_comment_message     string[python]
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
flag_order_multi_review              bool
dtype: object


In [209]:
df_products.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [210]:
#[IA] Criação de pipeline para verificação de products
def auditar_produtos(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add(nome, qtd, total=None, obs=""):
        total = len(df) if total is None else total
        checks.append({
            "check": nome,
            "problemas": int(qtd),
            "pct (%)": round(100 * qtd / total, 4) if total else 0.0,
            "status": "OK" if qtd == 0 else "FALHA",
            "obs": obs,
        })

    # ---------- 1. Nulos ----------
    for col in df.columns:
        add(f"nulos em {col}", df[col].isna().sum())

    # ---------- 2. IDs ----------
    for col in ["order_id", "product_id", "seller_id"]:
        s = df[col].astype(str)
        add(f"{col} vazio/whitespace", s.str.strip().eq("").sum())
        add(f"{col} com espaço nas bordas", (s != s.str.strip()).sum())

    # ---------- 3. order_item_id ----------
    add("order_item_id <= 0", (df["order_item_id"] <= 0).sum())

    # ---------- 4. Chave composta ----------
    add("PK (order_id, order_item_id) duplicada",
        df.duplicated(subset=["order_id", "order_item_id"]).sum())

    # ---------- 5. shipping_limit_date ----------
    dt = pd.to_datetime(df["shipping_limit_date"], errors="coerce")
    add("shipping_limit_date inválida", df["shipping_limit_date"].notna().mul(dt.isna()).sum())
    add("shipping_limit_date < 2000", (dt < DATA_MIN).sum())
    add("shipping_limit_date no futuro", (dt > pd.Timestamp.today()).sum())

    # ---------- 6. price ----------
    add("price < 0", (df["price"] < 0).sum())
    add("price == 0", (df["price"] == 0).sum(), obs="informativo (promo)")
    add("price com > 2 decimais",
        (~np.isclose(df["price"], df["price"].round(2), atol=1e-9)).sum())

    # ---------- 7. freight_value ----------
    add("freight_value < 0", (df["freight_value"] < 0).sum())
    add("freight_value == 0", (df["freight_value"] == 0).sum(), obs="informativo (frete grátis)")
    add("freight_value com > 2 decimais",
        (~np.isclose(df["freight_value"], df["freight_value"].round(2), atol=1e-9)).sum())

    return pd.DataFrame(checks)




In [211]:
#[IA] Criação de pipeline de limpeza de produtos
# Contudo, como visto abaixo, não há erros no dataset, portanto,
# não será necessário executar a limpeza.
def limpar_produtos(df: pd.DataFrame) -> pd.DataFrame:
    n0 = len(df)
    log = []
    df = df.copy().reset_index(drop=True)

    # --- IDs: strip ---
    for col in ["order_id", "product_id", "seller_id"]:
        df[col] = df[col].astype("string").str.strip()

    # --- order_item_id: coerce ---
    df["order_item_id"] = pd.to_numeric(df["order_item_id"], errors="coerce")

    # --- shipping_limit_date: datetime ---
    df["shipping_limit_date"] = pd.to_datetime(df["shipping_limit_date"], errors="coerce")

    # --- Numéricos: round 2 ---
    for col in ["price", "freight_value"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").round(2)

    # --- Máscara de invalidez dura (outlier NÃO entra — decisão do caderno) ---
    mask_ruim = (
        df["order_id"].isna() | df["order_id"].eq("")
        | df["product_id"].isna() | df["product_id"].eq("")
        | df["seller_id"].isna() | df["seller_id"].eq("")
        | df["order_item_id"].isna() | (df["order_item_id"] <= 0)
        | df["shipping_limit_date"].isna()
        | (df["shipping_limit_date"] < DATA_MIN)
        | (df["shipping_limit_date"] > pd.Timestamp.today())
        | df["price"].isna() | (df["price"] < 0)
        | df["freight_value"].isna() | (df["freight_value"] < 0)
    )

    n = int(mask_ruim.sum())
    if n:
        log.append(f"linhas removidas por invalidez dura: {n}")
    df = df[~mask_ruim].reset_index(drop=True)

    # --- Dedup pela PK composta ---
    antes = len(df)
    df = df.drop_duplicates(subset=["order_id", "order_item_id"], keep="last").reset_index(drop=True)
    if antes - len(df):
        log.append(f"PK (order_id, order_item_id) duplicada: {antes - len(df)}")

    # --- Cast final ---
    df["order_item_id"] = df["order_item_id"].astype("int32")
    df["price"] = df["price"].astype("float32")
    df["freight_value"] = df["freight_value"].astype("float32")

    print(f"Linhas: {n0} -> {len(df)} (removidas {n0-len(df)}, "
          f"{100*(n0-len(df))/n0:.3f}%)")
    for l in log:
        print("  •", l)
    return df



In [212]:
rel_produtos = auditar_produtos(df_products)
print(rel_items.to_string(index=False))
print(f"\nFalhas: {(rel_items['status']=='FALHA').sum()} | OK: {(rel_items['status']=='OK').sum()}")
#Como visto, não há falhas em df_products, além do falso positivo de freight_value ser grátis, que indica frete grátis.

                                 check  problemas  pct (%) status                        obs
                     nulos em order_id          0     0.00     OK                           
                nulos em order_item_id          0     0.00     OK                           
                   nulos em product_id          0     0.00     OK                           
                    nulos em seller_id          0     0.00     OK                           
          nulos em shipping_limit_date          0     0.00     OK                           
                        nulos em price          0     0.00     OK                           
                nulos em freight_value          0     0.00     OK                           
             order_id vazio/whitespace          0     0.00     OK                           
        order_id com espaço nas bordas          0     0.00     OK                           
           product_id vazio/whitespace          0     0.00     OK     